# AMDA catalogs with Speasy

<div align="center">
<img src="https://raw.githubusercontent.com/SciQLop/speasy/main/logo/logo_speasy.svg"/>
    <img src="http://cdpp.irap.omp.eu/images/logosOutils/amda.png" />
</div>

**What you'll learn**
- How to download an AMDA shared catalog with `speasy.get_data`.
- How to read catalog-level metadata and iterate over events.
- How to filter events and use them as time intervals to drive data fetches.

**Prerequisites** — [Speasy first steps](2-SpeasyFirstSteps.ipynb).

**Next** — [Speasy data manipulation](4-SpeasyDataManipulation.ipynb).


In [ ]:
import speasy as spz

# Downloading an AMDA catalog

A Speasy `Catalog` is essentially a list of time intervals (events) plus catalog-level metadata. They live in the AMDA inventory under `Catalogs/SharedCatalogs/...`.

We will download the LPP bowshock list, a catalog of identified bow shock crossings used widely in solar-wind studies.


In [ ]:
amda = spz.inventories.data_tree.amda
bs_catalog = spz.get_data(amda.Catalogs.SharedCatalogs.EARTH.LPP_bowshock_list)

print(f"{bs_catalog.name}: {len(bs_catalog)} events")
print(f"First event: {bs_catalog[0].start_time} → {bs_catalog[0].stop_time}")


## Reading catalog metadata

A shared catalog comes with provenance information: contact name, mission, time coverage, etc. It is exposed as a dictionary under `catalog.meta`.


In [ ]:
for k, v in bs_catalog.meta.items():
    print(f"{k}: {v}")


## Filtering events and plotting them

Each `Event` carries per-event metadata too (`event.meta`). The LPP bowshock catalog stores the crossing position in GSE; we'll filter for crossings near the X axis (\|Y\| < 0.2 Rₑ) and plot MMS1 data for each.

The helper below downloads the same three products as in tutorial 2 for a given event and saves a PNG to the working directory. It tries burst-mode data first and falls back to fast/survey if no burst data is available.


Below is a function that plots the omnidirectional energy flux, the ion bulk velocity and the magnetic field from MMS1, the same quantities as in exercice 2. : 

- mms1_dis_energyspectr_omni_brst
- mms1_dis_bulkv_gse_brst
- mms1_fgm_b_gse_brst_l2


In case there is no burst data for the event (hint: the len() of the time attribute is 0), then take the 

- mms1_dis_energyspectr_omni_fast
- mms1_dis_bulkv_gse_fast
- mms1_fgm_b_gse_srvy_l2

The following function takes a `dataset` that is a list of 3 `SpeasyVariable` for these quantities and saves figures on disk.
The goal of this exercice is to call this function so to make the same figure from exercice 2 for the events in the AMDA catalog just downloaded for which the Y coordinate of the crossing is such that $|Y| < 0.2 R_e$


In [ ]:
import matplotlib.pyplot as plt
%matplotlib widget
from matplotlib.colors import LogNorm


def plot_event(dataset):
    """Plot MMS1 omnidirectional energy flux, ion bulk velocity, and B-field for one event.

    `dataset` is a list of three SpeasyVariables in this order:
        [energy_flux_spectrogram, bulk_velocity, magnetic_field]
    """
    spectro, bulk_vel, magnetic = dataset

    fig, axes = plt.subplots(nrows=3, figsize=(10, 5), sharex=True)

    axes[0].pcolormesh(spectro.time,
                       spectro.axes[1].values[0, :],
                       spectro.values.T, cmap="jet", norm=LogNorm())
    axes[0].set_yscale("log")

    axes[1].plot(bulk_vel.time, bulk_vel.values, label=bulk_vel.columns)
    axes[2].plot(magnetic.time, magnetic.values, label=magnetic.columns)

    for ax in axes[1:]:
        ax.legend(ncol=4, loc="best")

    fig.tight_layout()
    fig.savefig(f"{spectro.time[0]}_{spectro.time[-1]}.png")
    plt.close(fig)


In [ ]:
import numpy as np

cda = spz.inventories.data_tree.cda

brst_products = [
    cda.MMS.MMS1.DIS.MMS1_FPI_BRST_L2_DIS_MOMS.mms1_dis_energyspectr_omni_brst,
    cda.MMS.MMS1.DIS.MMS1_FPI_BRST_L2_DIS_MOMS.mms1_dis_bulkv_gse_brst,
    cda.MMS.MMS1.FGM.MMS1_FGM_BRST_L2.mms1_fgm_b_gse_brst_l2,
]
fast_products = [
    cda.MMS.MMS1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_energyspectr_omni_fast,
    cda.MMS.MMS1.DIS.MMS1_FPI_FAST_L2_DIS_MOMS.mms1_dis_bulkv_gse_fast,
    cda.MMS.MMS1.FGM.MMS1_FGM_SRVY_L2.mms1_fgm_b_gse_srvy_l2,
]

for event in bs_catalog:
    # LPP bowshock catalog stores GSE coordinates in "Col 1" (X), "Col 2" (Y), "Col 3" (Z) in Re
    if np.abs(event.meta["Col 2"]) >= 0.2:
        continue

    dataset = spz.get_data(brst_products, event.start_time, event.stop_time)
    if dataset is None or len(dataset[0].time) == 0:
        dataset = spz.get_data(fast_products, event.start_time, event.stop_time)
    if dataset is None or len(dataset[0].time) == 0:
        continue

    plot_event(dataset)
